# Análise de Acidentes de Trânsito no Brasil

## 1. Introdução

Este notebook apresenta a análise exploratória de uma base simulada de acidentes de trânsito no Brasil entre 2015 e 2024.

O objetivo é identificar regiões críticas, períodos de maior risco, tipos de acidente, condições de visibilidade, gravidade, feridos e mortes.

As análises realizadas servem de base para o dashboard desenvolvido em Streamlit.


## 2. Contextualização do problema

Os acidentes de trânsito representam um problema de segurança pública e saúde.

A análise dos dados permite identificar padrões temporais e geográficos e investigar possíveis associações entre acidentes e variáveis como chuva, visibilidade e quantidade de veículos envolvidos.


## 3. Explicação da base

A base utilizada possui 4.440 registros e 15 colunas.

As principais variáveis são:

- `ano`: ano da ocorrência;
- `mes`: mês;
- `data`: data;
- `regiao`: região brasileira;
- `uf`: estado;
- `cidade`: cidade;
- `tipo_acidente`: tipo da ocorrência;
- `periodo_dia`: período do dia;
- `acidentes`: quantidade de acidentes;
- `feridos`: quantidade de feridos;
- `mortes`: quantidade de mortes;
- `chuva_mm`: volume de chuva em milímetros;
- `visibilidade`: condição de visibilidade;
- `veiculos_envolvidos`: quantidade de veículos;
- `nivel_gravidade`: nível de gravidade.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../dados/simulacao_acidentes_transito_brasil.csv")

df.head()


In [ ]:
print("Quantidade de linhas e colunas:", df.shape)

print("\nColunas:")
print(df.columns.tolist())

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isna().sum())

print("\nDuplicidades:")
print(df.duplicated().sum())


## 4. Limpeza e preparação

A data é convertida para `datetime` e as variáveis quantitativas são convertidas para tipos numéricos.

Também são removidas eventuais duplicidades.


In [ ]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")

colunas_numericas = [
    "ano",
    "mes",
    "acidentes",
    "feridos",
    "mortes",
    "chuva_mm",
    "veiculos_envolvidos"
]

for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df = df.drop_duplicates().copy()

display(df.head())


In [ ]:
display(df.describe(include="all").T)


## 5. Engenharia de atributos

São criadas variáveis auxiliares para facilitar a análise temporal e permitir análises agrupadas.


In [ ]:
df["ano_data"] = df["data"].dt.year
df["mes_data"] = df["data"].dt.month
df["dia_semana"] = df["data"].dt.day_name()

# Classificação simples do volume de chuva
df["faixa_chuva"] = pd.cut(
    df["chuva_mm"],
    bins=[-0.01, 0, 10, 30, float("inf")],
    labels=["Sem chuva", "Chuva leve", "Chuva moderada", "Chuva intensa"]
)

df.head()


## 6. Análise exploratória

### Evolução dos acidentes


In [ ]:
por_ano = (
    df.groupby("ano", as_index=False)["acidentes"]
    .sum()
)

display(por_ano)


In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(
    data=por_ano,
    x="ano",
    y="acidentes",
    marker="o"
)

plt.title("Evolução dos acidentes por ano")
plt.xlabel("Ano")
plt.ylabel("Acidentes")
plt.show()


### Comparação entre estados


In [ ]:
por_uf = (
    df.groupby("uf", as_index=False)["acidentes"]
    .sum()
    .sort_values("acidentes", ascending=False)
)

display(por_uf)


In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=por_uf,
    x="acidentes",
    y="uf"
)

plt.title("Acidentes por estado")
plt.xlabel("Acidentes")
plt.ylabel("Estado")
plt.show()


### Comparação entre regiões


In [ ]:
por_regiao = (
    df.groupby("regiao", as_index=False)["acidentes"]
    .sum()
    .sort_values("acidentes", ascending=False)
)

display(por_regiao)


In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=por_regiao,
    x="regiao",
    y="acidentes"
)

plt.title("Acidentes por região")
plt.xlabel("Região")
plt.ylabel("Acidentes")
plt.xticks(rotation=20)
plt.show()


### Tipos de acidente


In [ ]:
por_tipo = (
    df.groupby("tipo_acidente", as_index=False)["acidentes"]
    .sum()
    .sort_values("acidentes", ascending=False)
)

display(por_tipo)


In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=por_tipo,
    x="acidentes",
    y="tipo_acidente"
)

plt.title("Tipos de acidente mais frequentes")
plt.xlabel("Acidentes")
plt.ylabel("Tipo de acidente")
plt.show()


### Horário e mês


In [ ]:
heatmap = pd.pivot_table(
    df,
    values="acidentes",
    index="mes",
    columns="periodo_dia",
    aggfunc="sum",
    fill_value=0
)

plt.figure(figsize=(10, 5))

sns.heatmap(
    heatmap,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd"
)

plt.title("Acidentes por mês e período do dia")
plt.xlabel("Período do dia")
plt.ylabel("Mês")
plt.show()


### Condições de visibilidade


In [ ]:
por_visibilidade = (
    df.groupby("visibilidade", as_index=False)["acidentes"]
    .sum()
    .sort_values("acidentes", ascending=False)
)

display(por_visibilidade)


In [ ]:
plt.figure(figsize=(7, 7))

plt.pie(
    por_visibilidade["acidentes"],
    labels=por_visibilidade["visibilidade"],
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Acidentes por condição de visibilidade")
plt.show()


### Gravidade


In [ ]:
por_gravidade = (
    df.groupby("nivel_gravidade", as_index=False)
    .agg(
        acidentes=("acidentes", "sum"),
        feridos=("feridos", "sum"),
        mortes=("mortes", "sum")
    )
    .sort_values("acidentes", ascending=False)
)

display(por_gravidade)


In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=por_gravidade,
    x="nivel_gravidade",
    y="acidentes"
)

plt.title("Acidentes por nível de gravidade")
plt.xlabel("Gravidade")
plt.ylabel("Acidentes")
plt.show()


## 7. KPIs


In [ ]:
total_acidentes = df["acidentes"].sum()
total_feridos = df["feridos"].sum()
total_mortes = df["mortes"].sum()

estado_critico = (
    df.groupby("uf")["acidentes"]
    .sum()
    .idxmax()
)

periodo_critico = (
    df.groupby("periodo_dia")["acidentes"]
    .sum()
    .idxmax()
)

tipo_frequente = (
    df.groupby("tipo_acidente")["acidentes"]
    .sum()
    .idxmax()
)

print("Total de acidentes:", total_acidentes)
print("Total de feridos:", total_feridos)
print("Total de mortes:", total_mortes)
print("Estado mais crítico:", estado_critico)
print("Período mais perigoso:", periodo_critico)
print("Tipo mais frequente:", tipo_frequente)


## 8. Séries temporais avançadas

A média móvel de três meses é utilizada para observar a tendência dos acidentes, reduzindo o efeito de oscilações pontuais.


In [ ]:
mensal = (
    df.set_index("data")
    .resample("ME")
    .agg({
        "acidentes": "sum",
        "feridos": "sum",
        "mortes": "sum"
    })
    .reset_index()
)

mensal["media_movel_3m"] = (
    mensal["acidentes"]
    .rolling(3)
    .mean()
)

display(mensal.head(10))


In [ ]:
plt.figure(figsize=(12, 5))

sns.lineplot(
    data=mensal,
    x="data",
    y="acidentes",
    label="Acidentes"
)

sns.lineplot(
    data=mensal,
    x="data",
    y="media_movel_3m",
    label="Média móvel de 3 meses",
    linewidth=3
)

plt.title("Série temporal com média móvel")
plt.xlabel("Data")
plt.ylabel("Acidentes")
plt.show()


## 9. Correlação estatística

A matriz abaixo mostra a correlação linear entre acidentes, feridos, mortes, chuva e veículos envolvidos.

Correlação não significa causalidade.


In [ ]:
variaveis = [
    "acidentes",
    "feridos",
    "mortes",
    "chuva_mm",
    "veiculos_envolvidos"
]

correlacao = df[variaveis].corr()

display(correlacao)


In [ ]:
plt.figure(figsize=(9, 6))

sns.heatmap(
    correlacao,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Matriz de correlação")
plt.show()


## 10. Ranking de cidades críticas


In [ ]:
ranking_cidades = (
    df.groupby("cidade", as_index=False)
    .agg(
        acidentes=("acidentes", "sum"),
        feridos=("feridos", "sum"),
        mortes=("mortes", "sum")
    )
    .sort_values("acidentes", ascending=False)
)

display(ranking_cidades.head(15))


## 11. Interpretação dos resultados

Os resultados devem ser interpretados considerando o conjunto de dados e o recorte analisado.

Os principais pontos são o estado com maior concentração de acidentes, o período do dia mais frequente, o tipo de acidente predominante, a evolução temporal e as diferenças entre níveis de gravidade.

A análise de chuva e visibilidade permite investigar associações entre condições ambientais e ocorrências, enquanto a correlação estatística mostra relações lineares entre as variáveis quantitativas.


## 12. Conclusão

A análise da base permite transformar os registros de acidentes em informações úteis para compreender padrões temporais, geográficos e relacionados às características das ocorrências.

O dashboard Streamlit utiliza essas análises para oferecer filtros, KPIs, gráficos e tabelas interativas.

Foram utilizadas duas funcionalidades avançadas previstas na atividade: séries temporais avançadas e correlação estatística.
